# CTRL02 — Explicit Dependency Positive Control

C05 succeeded at accessibility but failed to manipulate utilization because the
generator had to compose label → city → routing rule → protocol while also
obeying a rigid output format.

C06 removes the composition step.

Each distant binding directly maps a nonce label to one protocol token:

\[
\text{JAPEV} \rightarrow \text{CHARLIE}.
\]

The same complete distant state is available in both conditions.

- `use_prior`: emit the protocol assigned in the distant binding table.
- `use_counterfactual`: emit an explicitly supplied deranged replacement mapping.

Both mappings use ALPHA/BRAVO/CHARLIE/DELTA exactly once.

The causal question is therefore simply:

\[
\boxed{\text{which mapping determines the emitted protocol token?}}
\]

Primary trace: \(U(k)\) measured specifically at the four protocol decision tokens.


In [ ]:
from pathlib import Path
import gc, json, random, itertools, re

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "experiments" else cwd
OUT = ROOT / "results" / "controls" / "CTRL02_explicit_dependency_positive_control"
OUT.mkdir(parents=True, exist_ok=True)

GENERATOR_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SCORER_ID = "Qwen/Qwen3-0.6B-Base"

SEED = 20260901
N = 60
PROTOCOLS = ["ALPHA", "BRAVO", "CHARLIE", "DELTA"]
CONDITIONS = ["use_prior", "use_counterfactual"]
WINDOWS = [8, 32, 128, 256]

rng = np.random.default_rng(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

ALPH = np.array(list("ABCDEFGHJKLMNPQRSTUVWXYZ"))


/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Controlled distant protocol bindings

In [2]:
PREAMBLE = """
This field report records operational assignments that remain in force unless
explicitly changed. Later sections should preserve previously recorded protocol
assignments for each equipment label. Routine inspection details do not alter
those assignments.

The reporting system also preserves a chronological administrative record before
operational assignments are entered. Inspectors verify the report identifier,
equipment manifest, communication status, transport schedule, access permissions,
battery inventory, protective cases, radio checks, weather covers, and routine
safety documentation. These administrative observations are intentionally
independent of any later protocol assignment. They establish document history but
do not determine which protocol belongs to any equipment label.

Before assignments are recorded, the team additionally confirms that identifiers
are unique, timestamps are synchronized, inventory counts are complete, and no
earlier temporary instruction remains active. This neutral setup information is
retained only so that later readers can reconstruct the report chronology. It
must not alter, predict, or replace any subsequent label-to-protocol mapping.
""".strip()

BRIDGE = """
After recording the protocol assignments, the team performed an unrelated
equipment inspection. Battery housings, cable connectors, protective cases,
radio units, seals, weather covers, and backup power modules were checked.
Nothing in this inspection changed any earlier protocol assignment.
""".strip()

def rlabel():
    return "".join(rng.choice(ALPH, size=5, replace=True).tolist())

def rcode(n=6):
    chars = np.array(list("ABCDEFGHJKLMNPQRSTUVWXYZ23456789"))
    return "".join(rng.choice(chars, size=n, replace=True).tolist())

def make_tail():
    return (
        f"Administrative reference {rcode(7)} was logged. "
        f"Inventory contained {int(rng.integers(8,35))} numbered tags and "
        f"{int(rng.integers(5,20))} sealed cases. "
        f"Radio verification code {rcode(4)} passed. "
        "These details are unrelated to protocol assignments."
    )

def make_instance(i):
    labels = []
    while len(labels) < 4:
        x = rlabel()
        if x not in labels:
            labels.append(x)
    labels = sorted(labels)

    true_perm = rng.permutation(PROTOCOLS).tolist()
    true_map = dict(zip(labels, true_perm))

    valid_cf = []
    for p in itertools.permutations(PROTOCOLS):
        cand = dict(zip(labels, p))
        if all(cand[lab] != true_map[lab] for lab in labels):
            valid_cf.append(cand)

    cf_map = valid_cf[int(rng.integers(0, len(valid_cf)))]

    binding_block = (
        "Fixed protocol assignments:\n"
        + "\n".join(f"{lab} -> {true_map[lab]}" for lab in labels)
        + "\nThese assignments remain active."
    )

    tail = make_tail()
    prior = PREAMBLE + "\n\n" + binding_block + "\n\n" + BRIDGE + "\n\n" + tail

    return {
        "instance_id": i,
        "labels": labels,
        "true_map": true_map,
        "cf_map": cf_map,
        "binding_block": binding_block,
        "tail": tail,
        "prior_document": prior,
    }

design = [make_instance(i) for i in range(N)]

display(pd.DataFrame([
    {
        "instance_id": d["instance_id"],
        "true_map": json.dumps(d["true_map"]),
        "cf_map": json.dumps(d["cf_map"]),
    }
    for d in design[:5]
]))


,instance_id,true_map,cf_map
0,0,"{""JAPEV"": ""DELTA"", ""JYGAD"": ""BRAVO"", ""LPBTZ"": ...","{""JAPEV"": ""CHARLIE"", ""JYGAD"": ""DELTA"", ""LPBTZ""..."
1,1,"{""AGAYU"": ""CHARLIE"", ""ELGBL"": ""ALPHA"", ""GHTWF""...","{""AGAYU"": ""DELTA"", ""ELGBL"": ""CHARLIE"", ""GHTWF""..."
2,2,"{""KRNYA"": ""DELTA"", ""LHTFC"": ""BRAVO"", ""LZTJF"": ...","{""KRNYA"": ""ALPHA"", ""LHTFC"": ""CHARLIE"", ""LZTJF""..."
3,3,"{""GVXXZ"": ""CHARLIE"", ""JESHZ"": ""ALPHA"", ""RHCYF""...","{""GVXXZ"": ""BRAVO"", ""JESHZ"": ""DELTA"", ""RHCYF"": ..."
4,4,"{""ARLLD"": ""ALPHA"", ""RRSUQ"": ""CHARLIE"", ""XKGYE""...","{""ARLLD"": ""DELTA"", ""RRSUQ"": ""ALPHA"", ""XKGYE"": ..."


## 2. Load generator and audit geometry

In [3]:
gtok = AutoTokenizer.from_pretrained(GENERATOR_ID, trust_remote_code=True)
gmodel = AutoModelForCausalLM.from_pretrained(
    GENERATOR_ID,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)
gmodel.eval()

SYSTEM_USER = (
    "You are maintaining a field report. Treat the previous assistant message "
    "as the report written so far and follow the new instruction exactly."
)

def enc(text):
    return gtok(
        text,
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids[0].cpu()

geom = []

for d in design:
    before = PREAMBLE + "\n\n"
    through = before + d["binding_block"]
    plen = len(enc(d["prior_document"]))
    st = len(enc(before))
    en = len(enc(through))

    geom.append({
        "instance_id": d["instance_id"],
        "prior_tokens": plen,
        "binding_start": st,
        "binding_end": en,
        "binding_end_to_cont": plen - en,
        "binding_start_to_cont": plen - st,
    })

geom = pd.DataFrame(geom)
display(geom.describe().T)

assert (geom["prior_tokens"] >= 256).all(), (
    "Prior is still too short for U(256). Increase only the neutral PREAMBLE; "
    "do not lengthen BRIDGE or TAIL, because that would move the causal binding farther away."
)
assert (geom["binding_end_to_cont"] > 32).all()
assert (geom["binding_end_to_cont"] < 160).all()

print("geometry checks passed")


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 338/338 [00:00<00:00, 586.09it/s]


,count,mean,std,min,25%,50%,75%,max
instance_id,60.0,29.500000,17.464249,0.0,14.75,29.5,44.25,59.0
prior_tokens,60.0,328.333333,1.945588,325.0,326.75,329.0,330.00,334.0
binding_start,60.0,199.000000,0.000000,199.0,199.00,199.0,199.00,199.0
binding_end,60.0,236.883333,0.940459,234.0,236.00,237.0,237.00,239.0
binding_end_to_cont,60.0,91.450000,1.598993,88.0,90.00,92.0,92.00,95.0
binding_start_to_cont,60.0,129.333333,1.945588,126.0,127.75,130.0,131.00,135.0


geometry checks passed


## 3. Exact decision queries and input-length matching

In [4]:
PAD_TEXT = (
    "Administrative note. Context unchanged. Reference only. "
    "No protocol assignment is modified. "
) * 16
PAD_IDS = enc(PAD_TEXT)

def prior_query(d):
    labels = ", ".join(d["labels"])
    return (
        "Write exactly four lines, one per label, in this alphabetical order:\n\n"
        + labels
        + "\n\nFor each line, copy the protocol previously assigned to that label "
          "in the report.\n\nUse exactly this format:\nLABEL PROTOCOL.\n\n"
          "The protocol must be exactly one of ALPHA, BRAVO, CHARLIE, DELTA. "
          "Write nothing else."
    )

def cf_query(d):
    mapping = "\n".join(
        f"{lab} -> {d['cf_map'][lab]}"
        for lab in d["labels"]
    )
    labels = ", ".join(d["labels"])

    return (
        "For this counterfactual output, ignore the earlier protocol assignments "
        "when choosing the protocol token.\n\nUse this replacement mapping exactly:\n\n"
        + mapping
        + "\n\nWrite exactly four lines, one per label, in this alphabetical order:\n\n"
        + labels
        + "\n\nUse exactly this format:\nLABEL PROTOCOL.\n\n"
          "The protocol must be exactly one of ALPHA, BRAVO, CHARLIE, DELTA. "
          "Write nothing else."
    )

def base_chat_ids(history, query):
    msgs = [
        {"role": "user", "content": SYSTEM_USER},
        {"role": "assistant", "content": history},
        {"role": "user", "content": query},
    ]
    x = gtok.apply_chat_template(
        msgs,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    if hasattr(x, "input_ids"):
        x = x.input_ids
    return x[0].cpu()

def matched_ids(d, condition):
    a = base_chat_ids(d["prior_document"], prior_query(d))
    b = base_chat_ids(d["prior_document"], cf_query(d))

    target_len = max(len(a), len(b))
    ids = a if condition == "use_prior" else b
    pad_n = target_len - len(ids)

    if pad_n > 0:
        assert pad_n <= len(PAD_IDS)
        ids = torch.cat([PAD_IDS[:pad_n], ids], dim=0)

    assert len(ids) == target_len
    return ids, pad_n

audit = []
for d in design:
    a, ap = matched_ids(d, "use_prior")
    b, bp = matched_ids(d, "use_counterfactual")
    assert len(a) == len(b)

    audit.append({
        "instance_id": d["instance_id"],
        "matched_input_tokens": len(a),
        "prior_pad_tokens": ap,
        "cf_pad_tokens": bp,
    })

audit = pd.DataFrame(audit)
display(audit.describe().T)
print(audit[["prior_pad_tokens","cf_pad_tokens"]].drop_duplicates())


,count,mean,std,min,25%,50%,75%,max
instance_id,60.0,29.500000,17.464249,0.0,14.75,29.5,44.25,59.0
matched_input_tokens,60.0,507.983333,3.357217,502.0,505.00,508.0,510.00,518.0
prior_pad_tokens,60.0,36.883333,0.940459,34.0,36.00,37.0,37.00,39.0
cf_pad_tokens,60.0,0.000000,0.000000,0.0,0.00,0.0,0.00,0.0


    prior_pad_tokens  cf_pad_tokens
0                 36              0
2                 38              0
4                 37              0
10                39              0
36                35              0
37                34              0


## 4. Accessibility check — STOP GATE 1

In [5]:
def greedy(ids, max_new_tokens=40):
    ids = ids.reshape(1,-1).to(gmodel.device)
    with torch.inference_mode():
        y = gmodel.generate(
            input_ids=ids,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            pad_token_id=gtok.eos_token_id,
            eos_token_id=gtok.eos_token_id,
            use_cache=True,
        )
    new = y[0, ids.shape[1]:]
    return gtok.decode(new, skip_special_tokens=True).strip(), new.detach().cpu()

ACCESS_PATH = OUT / "access_probes.parquet"

existing = pd.read_parquet(ACCESS_PATH) if ACCESS_PATH.exists() else pd.DataFrame()
done = set(zip(existing["instance_id"], existing["label"])) if len(existing) else set()
rows = existing.to_dict("records") if len(existing) else []

for d in tqdm(design, desc="C06 access probes"):
    for lab, protocol in d["true_map"].items():
        key = (d["instance_id"], lab)
        if key in done:
            continue

        q = f"What protocol was assigned to {lab}? Reply with the protocol word only."
        ids = base_chat_ids(d["prior_document"], q)
        ans, _ = greedy(ids, 8)

        rows.append({
            "instance_id": d["instance_id"],
            "label": lab,
            "expected_protocol": protocol,
            "answer": ans,
            "correct": protocol.lower() in ans.lower(),
        })

    pd.DataFrame(rows).to_parquet(ACCESS_PATH, index=False)

access = pd.read_parquet(ACCESS_PATH)

Z_access = (
    access.groupby("instance_id")["correct"]
    .mean()
    .rename("Z_access")
    .reset_index()
)

display(
    Z_access["Z_access"]
    .agg(["mean","std","count"])
    .to_frame()
    .T
)


C06 access probes: 100%|████████████████████████████████████████████████████████████████████████████████| 60/60 [00:33<00:00,  1.78it/s]


,mean,std,count
Z_access,0.9625,0.090021,60.0


Proceed only if \(Z_{\rm access}\) is high.


## 5. Generate both causal decision conditions

In [6]:
GEN_PATH = OUT / "continuations.parquet"

existing = pd.read_parquet(GEN_PATH) if GEN_PATH.exists() else pd.DataFrame()
done = set(zip(existing["instance_id"], existing["condition"])) if len(existing) else set()
rows = existing.to_dict("records") if len(existing) else []

for d in tqdm(design, desc="C06 continuations"):
    for condition in CONDITIONS:
        key = (d["instance_id"], condition)
        if key in done:
            continue

        ids, pad_n = matched_ids(d, condition)
        cont, new = greedy(ids, 80)

        rows.append({
            "instance_id": d["instance_id"],
            "condition": condition,
            "true_map_json": json.dumps(d["true_map"]),
            "cf_map_json": json.dumps(d["cf_map"]),
            "prior_document": d["prior_document"],
            "continuation_text": cont,
            "scored_document": d["prior_document"] + "\n\n" + cont,
            "input_tokens": len(ids),
            "prefix_pad_tokens": pad_n,
            "continuation_tokens": len(new),
        })

    pd.DataFrame(rows).to_parquet(GEN_PATH, index=False)

gens = pd.read_parquet(GEN_PATH)

display(
    gens.groupby("condition")[["input_tokens","continuation_tokens"]]
    .agg(["mean","std","count"])
)
display(
    gens.groupby("condition")["continuation_text"]
    .nunique()
    .rename("n_unique")
    .to_frame()
)


C06 continuations: 100%|████████████████████████████████████████████████████████████████████████████████| 60/60 [01:06<00:00,  1.10s/it]


input_tokens                 continuation_tokens            \
                           mean       std count                mean       std   
condition                                                                       
use_counterfactual   507.983333  3.357217    60           18.566667  7.404457   
use_prior            507.983333  3.357217    60           22.400000  5.848946   

                          
                   count  
condition                 
use_counterfactual    60  
use_prior             60

,n_unique
condition,
use_counterfactual,45
use_prior,50


## 6. Exact utilization parse — STOP GATE 2

In [7]:
def parse_mapping(row):
    true_map = json.loads(row["true_map_json"])
    cf_map = json.loads(row["cf_map_json"])
    text = row["continuation_text"]

    found = {}

    for lab in true_map:
        lines = [
            line.strip()
            for line in text.splitlines()
            if lab.lower() in line.lower()
        ]

        protocols = []
        for line in lines:
            for p in PROTOCOLS:
                if re.search(rf"\b{p}\b", line, flags=re.IGNORECASE):
                    protocols.append(p)

        protocols = list(dict.fromkeys(protocols))
        found[lab] = protocols[0] if len(protocols) == 1 else None

    parsed = [found[lab] is not None for lab in true_map]
    true_hits = [found[lab] == true_map[lab] for lab in true_map]
    cf_hits = [found[lab] == cf_map[lab] for lab in true_map]

    return pd.Series({
        "Z_parse": float(np.mean(parsed)),
        "Z_prior": float(np.mean(true_hits)),
        "Z_counterfactual": float(np.mean(cf_hits)),
    })

parsed = gens.apply(parse_mapping, axis=1)
gens = pd.concat(
    [gens.reset_index(drop=True), parsed.reset_index(drop=True)],
    axis=1,
)
gens.to_parquet(GEN_PATH, index=False)

display(
    gens.groupby("condition")[["Z_parse","Z_prior","Z_counterfactual"]]
    .agg(["mean","std","count"])
)

print("\nInstance 0")
for _, r in gens[gens["instance_id"] == 0].iterrows():
    print("\n---", r["condition"], "---")
    print(r["continuation_text"])
    print(
        "prior=", r["Z_prior"],
        "cf=", r["Z_counterfactual"],
        "parse=", r["Z_parse"],
    )


Z_parse                   Z_prior                  \
                        mean       std count      mean       std count   
condition                                                                
use_counterfactual  0.391667  0.485743    60  0.016667  0.129099    60   
use_prior           0.633333  0.461360    60  0.495833  0.454478    60   

                   Z_counterfactual                  
                               mean       std count  
condition                                            
use_counterfactual         0.375000  0.481655    60  
use_prior                  0.029167  0.093106    60


Instance 0

--- use_prior ---
JAPEV DELTA  
JYGAD BRAVO  
LPBTZ ALFA  
URVPU CHARLIE
prior= 0.75 cf= 0.0 parse= 0.75

--- use_counterfactual ---
CHARLIE, DELTA, BRAVO, ALPHA
prior= 0.0 cf= 0.0 parse= 0.0


Proceed only if both conditions have \(Z_{\rm parse}\approx1\), with
`use_prior` high on \(Z_{\rm prior}\) and `use_counterfactual` high on
\(Z_{\rm counterfactual}\).


## 7. Load frozen scorer and locate decision tokens

In [8]:
del gmodel
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

stok = AutoTokenizer.from_pretrained(SCORER_ID, trust_remote_code=True)
smodel = AutoModelForCausalLM.from_pretrained(
    SCORER_ID,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)
smodel.eval()

if stok.pad_token_id is None:
    stok.pad_token_id = stok.eos_token_id

def find_subseq(haystack, needle):
    hits = []
    n = len(needle)
    for i in range(len(haystack)-n+1):
        if haystack[i:i+n] == needle:
            hits.append(i)
    return hits

def locate_protocol_targets(prior, continuation):
    prior_ids = stok(prior, add_special_tokens=False).input_ids
    cont_ids = stok(continuation, add_special_tokens=False).input_ids
    start = len(prior_ids)

    targets = []

    for p in PROTOCOLS:
        starts = set()

        for surface in [p, " " + p]:
            needle = stok(surface, add_special_tokens=False).input_ids
            for h in find_subseq(cont_ids, needle):
                starts.add(h)

        if len(starts) != 1:
            return None

        targets.append(start + next(iter(starts)))

    return {
        "doc_ids": prior_ids + cont_ids,
        "targets": sorted(targets),
    }

loc_rows = []

for _, row in gens.iterrows():
    loc = locate_protocol_targets(
        row["prior_document"],
        row["continuation_text"],
    )

    loc_rows.append({
        "instance_id": row["instance_id"],
        "condition": row["condition"],
        "eligible": loc is not None,
        "n_targets": 0 if loc is None else len(loc["targets"]),
    })

loc_df = pd.DataFrame(loc_rows)

display(
    loc_df.groupby("condition")["eligible"]
    .agg(["sum","count"])
)


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████| 310/310 [00:00<00:00, 1441.11it/s]


,sum,count
condition,,
use_counterfactual,56,60
use_prior,49,60


## 8. Score only protocol decision tokens

In [9]:
SCORE_PATH = OUT / "scores.parquet"

def score_positions(doc_ids, targets):
    contexts = []
    target_ids = []
    ks = []

    for t in targets:
        for k in WINDOWS:
            if t < k:
                return None
            contexts.append(
                torch.tensor(doc_ids[t-k:t], dtype=torch.long)
            )
            target_ids.append(doc_ids[t])
            ks.append(k)

    max_len = max(len(x) for x in contexts)

    batch = torch.full(
        (len(contexts), max_len),
        stok.pad_token_id,
        dtype=torch.long,
    )
    mask = torch.zeros_like(batch)

    for i,x in enumerate(contexts):
        batch[i,-len(x):] = x
        mask[i,-len(x):] = 1

    batch = batch.to(smodel.device)
    mask = mask.to(smodel.device)
    tgt = torch.tensor(
        target_ids,
        dtype=torch.long,
        device=smodel.device,
    )

    with torch.inference_mode():
        logits = smodel(
            input_ids=batch,
            attention_mask=mask,
            use_cache=False,
        ).logits[:,-1,:]

        logp = torch.log_softmax(logits.float(), dim=-1)

        losses = -logp[
            torch.arange(len(tgt), device=smodel.device),
            tgt,
        ]

    losses = losses.detach().cpu().numpy()

    out = {}
    for k in WINDOWS:
        vals = [
            v for v,kk in zip(losses,ks)
            if kk == k
        ]
        out[f"L_{k}"] = float(np.mean(vals))

    out["U_32"] = out["L_8"] - out["L_32"]
    out["U_128"] = out["L_8"] - out["L_128"]
    out["U_256"] = out["L_8"] - out["L_256"]
    out["U_long"] = 0.5 * (out["U_128"] + out["U_256"])
    out["selectivity"] = out["U_long"] - out["U_32"]

    return out

existing = pd.read_parquet(SCORE_PATH) if SCORE_PATH.exists() else pd.DataFrame()
done = set(zip(existing["instance_id"], existing["condition"])) if len(existing) else set()
rows = existing.to_dict("records") if len(existing) else []

for _, row in tqdm(gens.iterrows(), total=len(gens), desc="C06 scoring"):
    key = (int(row["instance_id"]), str(row["condition"]))
    if key in done:
        continue

    loc = locate_protocol_targets(
        row["prior_document"],
        row["continuation_text"],
    )

    s = None if loc is None else score_positions(
        loc["doc_ids"],
        loc["targets"],
    )

    rows.append({
        "instance_id": int(row["instance_id"]),
        "condition": row["condition"],
        "eligible": s is not None,
        **({} if s is None else s),
    })

    pd.DataFrame(rows).to_parquet(SCORE_PATH, index=False)

scores = pd.read_parquet(SCORE_PATH)

display(
    scores.groupby("condition")["eligible"]
    .agg(["sum","count"])
)


C06 scoring: 100%|████████████████████████████████████████████████████████████████████████████████████| 120/120 [00:46<00:00,  2.58it/s]


,sum,count
condition,,
use_counterfactual,56,60
use_prior,49,60


## 9. Joint paired analysis and bootstrap

In [10]:
analysis = (
    gens.merge(Z_access, on="instance_id", how="left")
    .merge(scores, on=["instance_id","condition"], how="left")
)

elig = (
    analysis.pivot_table(
        index="instance_id",
        columns="condition",
        values="eligible",
        aggfunc="first",
    )
    .fillna(False)
)

joint = elig.index[elig[CONDITIONS].all(axis=1)]

a = analysis[analysis["instance_id"].isin(joint)].copy()

print("jointly eligible:", len(joint), "/", N)

display(
    a.groupby("condition")[[
        "Z_access","Z_parse","Z_prior","Z_counterfactual",
        "U_32","U_128","U_256","U_long","selectivity"
    ]].agg(["mean","std","count"])
)

def wide(metric):
    return (
        a.pivot(
            index="instance_id",
            columns="condition",
            values=metric,
        )[CONDITIONS]
        .dropna()
    )

def boot(metric, B=10000):
    w = wide(metric)
    d = (
        w["use_prior"]
        - w["use_counterfactual"]
    ).to_numpy(float)

    rr = np.random.default_rng(SEED)
    bs = np.empty(B)

    for i in range(B):
        ix = rr.integers(0, len(d), len(d))
        bs[i] = d[ix].mean()

    return {
        "metric": metric,
        "contrast": "use_prior - use_counterfactual",
        "n": len(d),
        "estimate": float(d.mean()),
        "ci_low": float(np.quantile(bs,.025)),
        "ci_high": float(np.quantile(bs,.975)),
    }

result = pd.DataFrame([
    boot(m)
    for m in [
        "Z_prior","Z_counterfactual",
        "U_32","U_128","U_256","U_long","selectivity"
    ]
])

display(result)
result.to_csv(OUT / "paired_contrasts.csv", index=False)


jointly eligible: 47 / 60


Z_access                  Z_parse                  \
                        mean      std count      mean       std count   
condition                                                               
use_counterfactual  0.968085  0.08433    47  0.340426  0.478975    47   
use_prior           0.968085  0.08433    47  0.680851  0.471186    47   

                     Z_prior                 Z_counterfactual  ... U_128  \
                        mean       std count             mean  ... count   
condition                                                      ...         
use_counterfactual  0.021277  0.145865    47         0.319149  ...    47   
use_prior           0.537234  0.471983    47         0.021277  ...    47   

                       U_256                    U_long                  \
                        mean       std count      mean       std count   
condition                                                                
use_counterfactual  3.508903  1.644440    47  3.114333  1.229325    47   
use_prior           5.821764  3.289982    47  5.382318  2.795347    47   

                   selectivity                  
                          mean       std count  
condition                                       
use_counterfactual    2.247821  1.080688    47  
use_prior             3.102167  2.615283    47  

[2 rows x 27 columns]

,metric,contrast,n,estimate,ci_low,ci_high
0,Z_prior,use_prior - use_counterfactual,47,0.515957,0.367021,0.659574
1,Z_counterfactual,use_prior - use_counterfactual,47,-0.297872,-0.436170,-0.159574
2,U_32,use_prior - use_counterfactual,47,1.413638,0.954858,1.854700
3,U_128,use_prior - use_counterfactual,47,2.223108,1.474636,2.973061
4,U_256,use_prior - use_counterfactual,47,2.312862,1.420481,3.210989
5,U_long,use_prior - use_counterfactual,47,2.267985,1.477067,3.055320
6,selectivity,use_prior - use_counterfactual,47,0.854346,0.064457,1.617420


In [11]:
# Exploratory exact-compliance subset.
# IMPORTANT: post-treatment diagnostic only, NOT the primary causal estimate.

gwide = gens.pivot(
    index="instance_id",
    columns="condition",
    values=["Z_parse", "Z_prior", "Z_counterfactual"],
)

exact_ids = gwide.index[
    (gwide[("Z_parse", "use_prior")] == 1.0)
    & (gwide[("Z_prior", "use_prior")] == 1.0)
    & (gwide[("Z_parse", "use_counterfactual")] == 1.0)
    & (gwide[("Z_counterfactual", "use_counterfactual")] == 1.0)
]

score_ok = scores.pivot(
    index="instance_id",
    columns="condition",
    values="eligible",
).fillna(False)

exact_ids = [
    i for i in exact_ids
    if i in score_ok.index
    and score_ok.loc[i, "use_prior"]
    and score_ok.loc[i, "use_counterfactual"]
]

print("Exact-compliance + jointly scoreable:", len(exact_ids), "/", N)

diag = (
    gens[gens["instance_id"].isin(exact_ids)]
    .merge(scores, on=["instance_id", "condition"])
)

display(
    diag.groupby("condition")[
        ["U_32", "U_128", "U_256", "U_long", "selectivity"]
    ].agg(["mean", "std", "count"])
)

if len(exact_ids) >= 5:
    w = diag.pivot(
        index="instance_id",
        columns="condition",
        values=["U_32", "U_128", "U_256", "U_long", "selectivity"],
    )

    rr = np.random.default_rng(SEED)
    rows = []

    for metric in ["U_32", "U_128", "U_256", "U_long", "selectivity"]:
        d = (
            w[(metric, "use_prior")]
            - w[(metric, "use_counterfactual")]
        ).to_numpy()

        bs = np.array([
            d[rr.integers(0, len(d), len(d))].mean()
            for _ in range(10000)
        ])

        rows.append({
            "metric": metric,
            "n": len(d),
            "estimate": d.mean(),
            "ci_low": np.quantile(bs, .025),
            "ci_high": np.quantile(bs, .975),
        })

    display(pd.DataFrame(rows))

Exact-compliance + jointly scoreable: 13 / 60


U_32                     U_128                  \
                        mean       std count      mean       std count   
condition                                                                
use_counterfactual  1.358212  1.479508    13  2.477994  1.613164    13   
use_prior           2.647689  0.734855    13  7.005970  1.799117    13   

                       U_256                    U_long                  \
                        mean       std count      mean       std count   
condition                                                                
use_counterfactual  4.992012  1.923437    13  3.735003  1.597955    13   
use_prior           8.731801  1.846689    13  7.868885  1.723229    13   

                   selectivity                  
                          mean       std count  
condition                                       
use_counterfactual    2.376791  1.093091    13  
use_prior             5.221197  1.838132    13

,metric,n,estimate,ci_low,ci_high
0,U_32,13,1.289476,0.501441,2.019102
1,U_128,13,4.527975,3.298398,5.712769
2,U_256,13,3.739789,2.366326,5.045772
3,U_long,13,4.133882,2.926425,5.216596
4,selectivity,13,2.844406,1.476464,3.919349


# Interpretation

If both causal rules are followed nearly perfectly, the experiment cleanly
changes which mapping determines protocol-token identity while holding the
available distant state fixed.

A resolved \(U\) contrast would show that this difference in causal parentage
survives into the completed-text context response.

A null \(U\) contrast after a successful utilization manipulation would be an
informative negative: even an explicit distant binding that causally determines
later token identity is not necessarily recoverable by this \(U\) operator.

C06 deliberately strips away natural prose and multi-hop reasoning. It tests:

\[
\boxed{
\text{distant explicit binding}
\rightarrow
\text{later token identity}
\rightarrow
U
}
\]
